In [ ]:
train_dir = '/kaggle/input/sgfood-train-test/datasets/train'
test_dir = '/kaggle/input/sgfood-train-test/datasets/test'

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
category_to_nutri_grade = {
    'Apple': 'A',
    'Apricot': 'A',
    'banana': 'A',
    'Blackberry': 'A',
    'blueberries': 'A',
    'Papaya': 'A',
    'orange': 'A',
    'pear': 'A',
    'salad': 'A',
    'mixed vegetables': 'A',
    'green leafy vegetables': 'A',
    'sandwich': 'A',
    'salmon - grilled': 'A',
    'Soft boiled eggs': 'A',
    'milk': 'A',
    'Nuts': 'A',
    'whole grain bread': 'A',
    'whole oats': 'A',
    'cooked brown rice': 'A',
    'cooked white rice': 'A',
    'corn': 'A',
    'Porridge': 'A',
    'yogurt': 'A',
    'thunder tea rice': 'A',

    'steamed grouper': 'B',
    'Ban Mian': 'B',
    'bee hoon': 'B',
    'Udon': 'B',
    'Fish Ball Noodles': 'B',
    'Seafood Noodles Soup': 'B',
    'Prawn Noodle': 'B',
    'sirloin steak': 'B',
    'pasta - red sauce': 'B',
    'dumpling': 'B',
    'siew mai': 'B',
    'Bibimbap': 'B',
    'chicken soup': 'B',
    'muesli': 'B',
    'popiah': 'B',
    'kebab - chicken': 'B',
    'sushi': 'B',
    'roasted chicken': 'B',
    'otak': 'B',

    'Lor mee': 'C',
    'Mee rebus': 'C',
    'Mee siam': 'C',
    'nasi lemak': 'C',
    'bak kut teh': 'C',
    'Duck Rice': 'C',
    'Claypot Rice': 'C',
    'rice dumpling': 'C',
    'pineapple tarts': 'C',
    'Miso ramen, with fishcake': 'C',
    'chwee kueh': 'C',
    'chicken rice': 'C',
    'Hor Fun': 'C',
    'hokkien prawn mee': 'C',
    'goreng pisang': 'C',
    'tacos and nachos': 'C',

    'Burger': 'D',
    'sambal stingray': 'D',
    'oyster omelette': 'D',
    'cheese fries': 'D',
    'bak kwa': 'D',
    'chilli crab': 'D',
    'black pepper crab': 'D',
    'fish head curry': 'D',
    'Indian Prata': 'D',
    'ayam penyet': 'D',
    'Kway Teow': 'D',
    'Fish and chips': 'D',
    'fried chicken': 'D',
    'har cheong gai': 'D',
    'satay bee hoon': 'D',
    'ice kacang': 'D',
    'Laksa': 'D',
    'Chinese fritters': 'D',
    'curry puff': 'D'
}

nutri_grade_to_numeric = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3
}

In [ ]:
import os
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

class EfficientNet(nn.Module):
    def __init__(self, class_no):
        super(EfficientNet, self).__init__()
        self.base = models.efficientnet_b0(pretrained=True)
        # fine-tune the last two layers
        for param in self.base.features[-2:].parameters():
            param.requires_grad = True
        self.base.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.base.classifier[1].in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, class_no)
        )
    def forward(self, x):
        return self.base(x)

In [ ]:
import torch
from torch.utils.data import Dataset

class NutriGradeDataset(Dataset):
    def __init__(self, dataset, category_to_nutri_grade, nutri_grade_to_numeric):
        self.dataset = dataset
        self.category_to_nutri_grade = category_to_nutri_grade
        self.nutri_grade_to_numeric = nutri_grade_to_numeric

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        class_name = self.dataset.classes[label]
        nutri_grade = self.category_to_nutri_grade[class_name]
        nutri_idx = self.nutri_grade_to_numeric[nutri_grade]

        return img, nutri_idx

In [ ]:
from torch.utils.data import DataLoader, random_split
def load_data(train_dir, test_dir, batch_size):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        # normalize to match with pretrained value
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    data = datasets.ImageFolder(train_dir, transform=transform)
    data_converted = NutriGradeDataset(data, category_to_nutri_grade, nutri_grade_to_numeric)

    val_size = int(0.2 * len(data_converted))
    train_size = len(data_converted) - val_size
    train_data, val_data = random_split(data_converted, [train_size, val_size])

    test_data = datasets.ImageFolder(test_dir, transform=transform)
    test_data_converted = NutriGradeDataset(test_data, category_to_nutri_grade, nutri_grade_to_numeric)
    train_loader = DataLoader(train_data, batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_data, batch_size, shuffle=False, num_workers=2)

    test_loader  = DataLoader(test_data_converted, batch_size)
    return train_loader, val_loader, test_loader

In [ ]:
import time
from torch.amp import autocast, GradScaler
def train(model, train_loader, val_loader, criterion, optimizer, epochs):
    scaler = GradScaler()
    min_val_loss = float('inf')
    patience = 2
    trigger_times = 0
    model.to(device)
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        training_loss = 0.0
        for i, l in train_loader:
            i, l = i.to(device), l.to(device)
            optimizer.zero_grad()
            # reduce precision for less mem & faster training
            with autocast(device_type='cuda'):
                output = model(i)
                loss = criterion(output, l)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            training_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for i, l in val_loader:
                i, l = i.to(device), l.to(device)
                with autocast(device_type='cuda'):
                    outputs = model(i)
                    loss = criterion(outputs, l)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        epoch_time = time.time() - start_time
        print(f'epoch [{epoch+1}/{epochs}], train loss: {training_loss/len(train_loader):.4f}  | val loss: {val_loss/len(val_loader):.4f}  | '
              f'time: {epoch_time:.2f}s')

        # early stopping
        if avg_val_loss < min_val_loss:
            min_val_loss = avg_val_loss
            best_model_wts = model.state_dict()
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("early stopping")
                break

    model.load_state_dict(best_model_wts)
    return model


In [ ]:
def evaluate(model, test_loader, criterion):
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0

    with torch.no_grad():
        for i, l in test_loader:
            i, l = i.to(device), l.to(device)
            output = model(i)
            loss = criterion(output, l)
            test_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += l.size(0)
            correct += (predicted == l).sum().item()

    accuracy = correct * 100 / total
    avg_loss = test_loss / len(test_loader)

    print(f'test loss: {avg_loss:.4f} | test accuracy: {accuracy:.4f}%')

In [ ]:
class_no = 4
train_loader, val_loader, test_loader = load_data(train_dir, test_dir, 64)
model = EfficientNet(class_no)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
train(model, train_loader, val_loader, criterion, optimizer, 10)
evaluate(model, test_loader, criterion)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 86.2MB/s]


epoch [1/10], train loss: 0.8631  | val loss: 0.6111  | time: 142.02s
epoch [2/10], train loss: 0.5094  | val loss: 0.5224  | time: 89.30s
epoch [3/10], train loss: 0.3428  | val loss: 0.5194  | time: 91.98s
epoch [4/10], train loss: 0.2393  | val loss: 0.5476  | time: 155.08s
epoch [5/10], train loss: 0.1701  | val loss: 0.5997  | time: 116.20s
early stopping
test loss: 0.5921 | test accuracy: 82.7752%


In [ ]:
torch.save(model.state_dict(), 'efficientNet_pytorch_4_model.pth')